# FAO2HKB — Reproducible Run Report

## Purpose
This notebook rebuilds a **FAO2HKB run from scratch** (download → build → validate → visualize → bundle) using **public FAOSTAT Bulk Downloads** and a **pinned code revision**.

It is designed for reviewers and readers to understand:
1) **Data provenance**: which public sources were used (explicit URLs in the config).
2) **Data product**: which artifacts are generated (data structures at multiple levels).
3) **Reproducibility evidence**: exact code commit, resolved config, logs, timings.
4) **Diagnostics**: series-level properties + optional embedding PCA maps.
5) **Human-readable examples**: 10 random time-series with “cards” (L3 → L2 → L1).
6) **Zenodo packaging**: one ZIP file ready to upload.

## Key outputs
After running end-to-end, you will have:
- `RUN_DIR/artifacts/data/{DOMAIN,SERIES,RECORDS}.jsonl`
- `RUN_DIR/work/execution/*` (CLI stdout, timings, milestones, run summary)
- `RUN_DIR/work/plots/diagnostics/*` (properties plots, PCA mapse)
- `RUN_DIR/work/random_series_cards.{txt,json}`
- `fao2hkb_bundle__<RUN_ID>.zip` (single upload artifact for Zenodo)

## How reviewers reproduce
Reviewers can reproduce by running this notebook top-to-bottom with:
- the **same `sources:` URLs** in the config,
- the **same GitHub repository + commit hash** recorded in `repo_commit.txt`.

> Tip: Run cells in order. If a cell fails, the printed diagnostics should indicate what to fix.


In [ ]:
#@title 0) Check environment (Python / OS / GPU)
"""
MDPI/Data descriptor note:
- Diagnostic only: this cell documents the execution context (Python, OS, GPU).
- Hardware can affect runtime (speed/memory) but should not change the produced data artifacts.
"""

import sys
import platform
import subprocess

# ---------------------------------------------------------------------
# 1) Basic runtime context
# ---------------------------------------------------------------------
print("Python   :", sys.version.split()[0])
print("Platform :", platform.platform())

# ---------------------------------------------------------------------
# 2) GPU context (best-effort)
#    Torch is used here only to detect CUDA availability.
# ---------------------------------------------------------------------
try:
    import torch
    print("Torch    :", torch.__version__)
    print("CUDA     :", torch.cuda.is_available())
except Exception as e:
    print("Torch import failed (ok for non-embedding runs):", type(e).__name__, str(e)[:160])

# ---------------------------------------------------------------------
# 3) NVIDIA driver info (Linux/Colab only; not fatal if missing)
# ---------------------------------------------------------------------
try:
    subprocess.run(["nvidia-smi"], check=False)
except Exception as e:
    print("nvidia-smi not available:", type(e).__name__, str(e)[:160])

print("\nNote: If CUDA is False and you expect a GPU, enable a GPU runtime and re-run this cell.")


Python   : 3.12.12
Platform : Linux-6.6.105+-x86_64-with-glibc2.35
Torch    : 2.9.0+cu126
CUDA     : True

Note: If CUDA is False and you expect a GPU, enable a GPU runtime and re-run this cell.


In [ ]:
#@title 1) Fetch code (clone + checkout commit)
"""
MDPI/Data descriptor note:
- Reproducibility requires pinning *code* as well as *data sources*.
- This cell fetches the FAO2HKB repository and checks out a chosen ref (branch or tag).
- We record the exact commit hash in `repo_commit.txt` so reviewers can re-run using the same code state.
- For strict reproducibility, prefer a release tag (e.g., `vX.Y.Z`) over a moving branch like `main`.
"""

from pathlib import Path
import os
import subprocess

# ---------------------------------------------------------------------
# 1) Reviewer-editable settings
#    (These are the only values a reviewer typically needs to change.)
# ---------------------------------------------------------------------
BASE_DIR = Path.cwd()  # Works in Colab (/content) and locally (current folder)
REPO_URL = "https://github.com/JacoboGGLeon/fao2hkb"
REPO_DIR = BASE_DIR / "fao2hkb"

# Recommended: set to a release tag (vX.Y.Z) for strict reproducibility
REPO_TAG = "main"

# ---------------------------------------------------------------------
# 2) Clone repository (only if missing)
# ---------------------------------------------------------------------
if not REPO_DIR.is_dir():
    subprocess.check_call(["git", "clone", REPO_URL, str(REPO_DIR)])

# ---------------------------------------------------------------------
# 3) Fetch refs and check out the requested tag/branch/commit
#    We run commands from inside REPO_DIR to avoid path ambiguity.
# ---------------------------------------------------------------------
subprocess.check_call(["bash", "-lc", f"cd {REPO_DIR} && git fetch --all --tags -q"])
subprocess.check_call(["bash", "-lc", f"cd {REPO_DIR} && git checkout -q {REPO_TAG}"])

# ---------------------------------------------------------------------
# 4) Record the exact commit hash (the true reproducibility anchor)
# ---------------------------------------------------------------------
commit = subprocess.check_output(["bash", "-lc", f"cd {REPO_DIR} && git rev-parse HEAD"]).decode().strip()

print("✅ repo  :", REPO_URL)
print("✅ ref   :", REPO_TAG)
print("✅ commit:", commit)

# Save provenance file (this will be included later in the Zenodo bundle)
repo_commit_path = BASE_DIR / "repo_commit.txt"
repo_commit_path.write_text(f"repo={REPO_URL}\nref={REPO_TAG}\ncommit={commit}\n", encoding="utf-8")
print("✅ saved:", str(repo_commit_path))


✅ repo  : https://github.com/JacoboGGLeon/fao2hkb
✅ ref   : main
✅ commit: 89403d28dfcfd54363835c6b407c651ae1a29f1a
✅ saved: /content/repo_commit.txt


In [ ]:
#@title 2) Install package + dependencies
"""
MDPI/Data descriptor note:
- Installs the FAO2HKB code (editable) and the runtime dependencies used by this notebook.
- Embeddings are optional: when enabled, we install extra dependencies and try FAISS (GPU if available, else CPU).
- We perform a small "import provenance" check to avoid shadow imports (common in notebooks).
"""

import sys, subprocess
from pathlib import Path
import importlib

# ---------------------------------------------------------------------
# 0) Upgrade pip (more reliable installs in notebooks and local venvs)
# ---------------------------------------------------------------------
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip"])

# ---------------------------------------------------------------------
# 1) User option (kept notebook-friendly)
# ---------------------------------------------------------------------
INSTALL_EMBEDDINGS = True  # @param {type:"boolean"}

# ---------------------------------------------------------------------
# 2) Install repo in editable mode (uses the checked-out code)
# ---------------------------------------------------------------------
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR])
print("✅ base installed (-e)")

if INSTALL_EMBEDDINGS:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO_DIR}[embeddings]"])
    print("✅ embeddings extra installed")

    # FAISS is optional: try GPU first, then CPU
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "faiss-gpu"])
        print("✅ faiss-gpu installed")
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "faiss-cpu"])
        print("✅ faiss-cpu installed")

# Common scientific/plotting stack
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "matplotlib", "tqdm", "pandas", "numpy", "scikit-learn", "pyyaml"
])
print("✅ plotting deps installed")

# ---------------------------------------------------------------------
# 3) Shadow-import prevention (src-layout safe)
#    In notebooks, the repo folder name can "win" over the real package.
# ---------------------------------------------------------------------
repo_dir = Path(REPO_DIR).resolve()
src_dir = repo_dir / "src"

# If this is a src-layout repo, prioritize /src to ensure we import the real package code.
if (src_dir / "fao2hkb").is_dir():
    src_str = str(src_dir)
    if src_str not in sys.path:
        sys.path.insert(0, src_str)

# Clear any previously-imported shadow/namespace version
for k in list(sys.modules.keys()):
    if k == "fao2hkb" or k.startswith("fao2hkb."):
        del sys.modules[k]

# ---------------------------------------------------------------------
# 4) Import provenance report (minimal + reviewer-friendly)
# ---------------------------------------------------------------------
import fao2hkb

pkg_file = getattr(fao2hkb, "__file__", None)
pkg_path = list(getattr(fao2hkb, "__path__", []))

if pkg_file:
    # Normal package: points to .../fao2hkb/__init__.py
    print("✅ Import check: fao2hkb loaded from:", pkg_file)
else:
    # Namespace package: no single __init__.py file; show search paths.
    # This can happen if the repo folder name matches the package name in a notebook context.
    print("⚠️ Import check: fao2hkb is a namespace package (no __file__).")
    print("   Search paths:", pkg_path)

✅ base installed (-e)
✅ embeddings extra installed
✅ faiss-cpu installed
✅ plotting deps installed
✅ Import check: fao2hkb loaded from: /content/fao2hkb/src/fao2hkb/__init__.py


In [ ]:
#@title 3) Define JSONL streaming helpers
"""
MDPI/Data descriptor note:
- Core artifacts are stored as JSONL (one JSON object per line) to scale to large datasets.
- These helpers stream files line-by-line to avoid loading entire artifacts into memory.
- They are used for diagnostics and a small reviewer-friendly demo subset.
"""

import json
import random
from typing import Any, Dict, Iterable, List, Set

# ---------------------------------------------------------------------
# 1) Minimal JSONL streaming
# ---------------------------------------------------------------------
def read_jsonl_iter(path: str) -> Iterable[Dict[str, Any]]:
    """
    Stream a JSONL file line-by-line.

    Why:
    - JSONL scales well for large artifacts.
    - Streaming keeps memory usage stable and reviewer-friendly.
    """
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                yield json.loads(line)

# ---------------------------------------------------------------------
# 2) Convenience loaders (small artifacts only)
# ---------------------------------------------------------------------
def load_domains(path: str) -> List[Dict[str, Any]]:
    """
    Load DOMAIN.jsonl into memory.

    DOMAIN.jsonl is typically small (metadata-level), so loading it fully is fine.
    """
    return list(read_jsonl_iter(path))

# ---------------------------------------------------------------------
# 3) Reviewer-friendly sampling utilities
# ---------------------------------------------------------------------
def reservoir_sample_series(path: str, k: int, seed: int = 42) -> List[Dict[str, Any]]:
    """
    Uniform random sampling from a JSONL stream without precomputing file length.

    This uses reservoir sampling:
    - Every series has equal probability to be selected.
    - Works even when the file is too large to load into memory.
    """
    rng = random.Random(int(seed))
    sample: List[Dict[str, Any]] = []
    n = 0

    for row in read_jsonl_iter(path):
        n += 1
        if len(sample) < int(k):
            sample.append(row)
        else:
            j = rng.randint(1, n)  # inclusive
            if j <= int(k):
                sample[j - 1] = row

    return sample

def scan_records_for_keys(path: str, keys_set: Set[str]) -> Dict[str, Dict[str, Any]]:
    """
    Scan RECORDS.jsonl until all requested series_keys are found.

    Notes:
    - This is intentionally simple and robust for small demo subsets.
    - For large-scale joins, the pipeline provides structured artifacts/indices instead.
    """
    out: Dict[str, Dict[str, Any]] = {}
    keys_set = set(keys_set)

    for row in read_jsonl_iter(path):
        sk = (row.get("identity", {}) or {}).get("series_key")
        if sk in keys_set:
            out[sk] = row
            if len(out) == len(keys_set):
                break

    return out

print("✅ helpers ready")


✅ helpers ready


## Configure → Freeze → Run (reproducible experiment)

In this section we define a minimal experiment and run it end-to-end.

**Why two config files?**
- `config.yaml` is the human-edited baseline.
- `config.resolved.yaml` is the *exact* file passed to the CLI and packaged for Zenodo.

**Reproducibility rules used here:**
- A timestamped `run_id` avoids collisions across runs.
- The resolved config records sources and pipeline settings explicitly.
- The notebook writes execution evidence (stdout, timings, milestones) into `RUN_DIR/work/execution/`.


In [ ]:
#@title 4a) Write base run config (config.yaml)
"""
MDPI/Data descriptor note:
- This cell writes a small, human-readable `config.yaml`.
- The most important part for reviewers is `sources:`: it explicitly lists the public FAOSTAT bulk ZIP URLs.
- Reviewers may change `sources:` to reproduce other tables, but URLs should remain public and stable.
- The next cell creates a *resolved* config that pins the exact run_id and consistency flags used in execution.
"""

from pathlib import Path

# ---------------------------------------------------------------------
# 1) Where we write the base config
# ---------------------------------------------------------------------
CONFIG_YAML_BASE = Path.cwd() / "config.yaml"

# ---------------------------------------------------------------------
# 2) Minimal config text (kept readable for reviewers)
#    Notes:
#    - run.run_id is intentionally generic here; the next cell timestamps it.
#    - embeddings are optional; if enabled, INSTALL_EMBEDDINGS must be True in the install cell.
# ---------------------------------------------------------------------
CONFIG_TEXT = """\
# FAO2HKB run config (minimal, reproducible)
#
# Edit `sources:` to add or change FAOSTAT tables (Normalized bulk ZIPs).
# IMPORTANT:
# - If embeddings.enabled: true, you MUST set INSTALL_EMBEDDINGS=True in the install cell above.

run:
  run_id: "FAO2HKB"          # overwritten by next cell (timestamped)
  output_root: "./runs"
  overwrite_downloads: false
  seed: 42

io:
  raw_dir: "raw"
  work_dir: "work"
  artifacts_dir: "artifacts"

build:
  include_codes: true
  drop_columns_regex: ".*(Source|Flag|Note|Unnamed).*"

embeddings:
  enabled: true
  levels: ["l3","l2"]
  model_name: "Qwen/Qwen3-Embedding-0.6B"
  device: "auto"
  max_seq_length: 1024
  batch_size: 2048
  l2_normalize: true

packaging:
  tar_gz: false
  include_raw: true

reproducibility:
  record_env: true
  record_pip_freeze: true

sources:
  - table: 1
    url: "https://bulks-faostat.fao.org/production/Production_Crops_Livestock_E_All_Data_(Normalized).zip"
    meta:
      Domain: "Production"
      Sub_domain: "Production"
      Domain_table: "Crops and livestock products"
  - table: 2
    url: "https://bulks-faostat.fao.org/production/Production_Indices_E_All_Data_(Normalized).zip"
    meta:
      Domain: "Production"
      Sub_domain: "Production"
      Domain_table: "Production Indices"
  - table: 3
    url: "https://bulks-faostat.fao.org/production/Value_of_Production_E_All_Data_(Normalized).zip"
    meta:
      Domain: "Production"
      Sub_domain: "Production"
      Domain_table: "Value of Agricultural Production"
  - table: 4
    url: "https://bulks-faostat.fao.org/production/Emissions_Totals_E_All_Data_(Normalized).zip"
    meta:
      Domain: "Climate Change: Agrifood systems emissions"
      Sub_domain: "Totals and Indicators"
      Domain_table: "Emissions totals"
  - table: 5
    url: "https://bulks-faostat.fao.org/production/Climate_change_Emissions_indicators_E_All_Data_(Normalized).zip"
    meta:
      Domain: "Climate Change: Agrifood systems emissions"
      Sub_domain: "Totals and Indicators"
      Domain_table: "Emissions indicators"
  - table: 6
    url: "https://bulks-faostat.fao.org/production/Environment_Emissions_intensities_E_All_Data_(Normalized).zip"
    meta:
      Domain: "Climate Change: Agrifood systems emissions"
      Sub_domain: "Totals and Indicators"
      Domain_table: "Emissions intensities"
  - table: 7
    url: "https://bulks-faostat.fao.org/production/Emissions_crops_E_All_Data_(Normalized).zip"
    meta:
      Domain: "Climate Change: Agrifood systems emissions"
      Sub_domain: "Farm gate"
      Domain_table: "Emissions from Crops"
  - table: 8
    url: "https://bulks-faostat.fao.org/production/Emissions_livestock_E_All_Data_(Normalized).zip"
    meta:
      Domain: "Climate Change: Agrifood systems emissions"
      Sub_domain: "Farm gate"
      Domain_table: "Emissions from Livestock"
  - table: 9
    url: "https://bulks-faostat.fao.org/production/Emissions_Agriculture_Energy_E_All_Data_(Normalized).zip"
    meta:
      Domain: "Climate Change: Agrifood systems emissions"
      Sub_domain: "Farm gate"
      Domain_table: "Emissions from Energy use in agriculture"
  - table: 10
    url: "https://bulks-faostat.fao.org/production/Emissions_Land_Use_Forests_E_All_Data_(Normalized).zip"
    meta:
      Domain: "Climate Change: Agrifood systems emissions"
      Sub_domain: "Land use and change"
      Domain_table: "Emissions from Forests"
  - table: 11
    url: "https://bulks-faostat.fao.org/production/Emissions_Land_Use_Fires_E_All_Data_(Normalized).zip"
    meta:
      Domain: "Climate Change: Agrifood systems emissions"
      Sub_domain: "Land use and change"
      Domain_table: "Emissions from Fires"
  - table: 12
    url: "https://bulks-faostat.fao.org/production/Emissions_Drained_Organic_Soils_E_All_Data_(Normalized).zip"
    meta:
      Domain: "Climate Change: Agrifood systems emissions"
      Sub_domain: "Land use and change"
      Domain_table: "Emissions from Drained organic soils"
  - table: 13
    url: "https://bulks-faostat.fao.org/production/Emissions_Pre_Post_Production_E_All_Data_(Normalized).zip"
    meta:
      Domain: "Climate Change: Agrifood systems emissions"
      Sub_domain: "Pre and post agricultural production"
      Domain_table: "Emissions from pre and post agricultural production"
"""

# ---------------------------------------------------------------------
# 3) Write the file (reviewers can open it easily)
# ---------------------------------------------------------------------
CONFIG_YAML_BASE.write_text(CONFIG_TEXT, encoding="utf-8")
print("✅ wrote:", str(CONFIG_YAML_BASE))


✅ wrote: /content/config.yaml


In [ ]:
#@title 4b) Freeze resolved config (config.resolved.yaml)
"""
MDPI/Data descriptor note:
- This cell produces `config.resolved.yaml`, the *exact* configuration passed to the FAO2HKB CLI.
- It pins a timestamped run_id to avoid collisions and to make the output folder self-identifying.
- It also enforces a simple reproducibility consistency rule:
  If embeddings are enabled in the config, the notebook must have INSTALL_EMBEDDINGS=True.
"""

import datetime
import os
from pathlib import Path

import yaml

# ---------------------------------------------------------------------
# 1) Load the base config written in the previous cell
# ---------------------------------------------------------------------
BASE_CONFIG_PATH = Path.cwd() / "config.yaml"
assert BASE_CONFIG_PATH.exists(), f"Missing: {BASE_CONFIG_PATH}"

cfg = yaml.safe_load(BASE_CONFIG_PATH.read_text(encoding="utf-8")) or {}

# ---------------------------------------------------------------------
# 2) Resolve run_id (timestamp) to avoid collisions and improve traceability
# ---------------------------------------------------------------------
ts = datetime.datetime.now(datetime.UTC).strftime("%Y%m%d_%H%M%S")
run_id_base = str((cfg.get("run") or {}).get("run_id") or "FAO2HKB")
cfg.setdefault("run", {})["run_id"] = f"{run_id_base}_{ts}"

# ---------------------------------------------------------------------
# 3) Normalize output_root (helps when moving between Colab and local runs)
# ---------------------------------------------------------------------
out_root = str(cfg["run"].get("output_root", "./runs") or "./runs").strip()
cfg["run"]["output_root"] = os.path.normpath(out_root)

# ---------------------------------------------------------------------
# 4) Expose SEED as a notebook-level variable for downstream sampling/plots
# ---------------------------------------------------------------------
SEED = int(cfg["run"].get("seed", 42))

# ---------------------------------------------------------------------
# 5) Enforce a simple "config ↔ environment" consistency rule
#    (Prevents a confusing situation for reviewers: embeddings enabled but deps not installed.)
# ---------------------------------------------------------------------
emb_enabled = bool((cfg.get("embeddings") or {}).get("enabled", False))
if emb_enabled and not bool(INSTALL_EMBEDDINGS):
    print("⚠️ embeddings.enabled=true but INSTALL_EMBEDDINGS=False → auto-disabling embeddings.")
    cfg.setdefault("embeddings", {})["enabled"] = False
    emb_enabled = False

# ---------------------------------------------------------------------
# 6) Write resolved config (this is what we pass to the CLI)
# ---------------------------------------------------------------------
CONFIG_PATH = str(Path.cwd() / "config.resolved.yaml")
Path(CONFIG_PATH).write_text(
    yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True),
    encoding="utf-8",
)

# ---------------------------------------------------------------------
# 7) Human-readable summary for reviewers
# ---------------------------------------------------------------------
print("✅ CONFIG_PATH =", CONFIG_PATH)
print("✅ RUN_ID      =", cfg["run"]["run_id"])
print("✅ output_root =", cfg["run"]["output_root"])
print("✅ seed        =", SEED)
print("✅ embeddings  =", emb_enabled)
print("✅ sources     =", len(cfg.get("sources", []) or []))


✅ CONFIG_PATH = /content/config.resolved.yaml
✅ RUN_ID      = FAO2HKB_20260107_181757
✅ output_root = runs
✅ seed        = 42
✅ embeddings  = True
✅ sources     = 13


In [ ]:
#@title 4c) Run pipeline (CLI) + capture outputs (JSON) + save evidence
"""
MDPI/Data descriptor note:
- This cell executes the FAO2HKB pipeline using the resolved configuration (`config.resolved.yaml`).
- It captures *evidence* of execution for reviewers:
  - full CLI stdout (cleaned from ANSI codes),
  - the final JSON summary emitted by the CLI,
  - a timing summary (wall time, number of log lines),
  - a preview of the pipeline execution milestones (if available).
- It also resolves and validates the expected output artifacts (DOMAIN/SERIES/RECORDS JSONL paths).
"""

import json
import os
import re
import subprocess
import time

from tqdm.auto import tqdm

# ---------------------------------------------------------------------
# 1) Execute the CLI (run from the repo directory for consistent behavior)
# ---------------------------------------------------------------------
# NOTE: We use "bash -lc" so "cd" works reliably across Colab/Linux/macOS.
# On Windows, reviewers should run this notebook in WSL or adapt the command.
cmd = f'cd "{REPO_DIR}" && python -m fao2hkb run --config "{CONFIG_PATH}"'
print("Running:", cmd)

t0 = time.perf_counter()

proc = subprocess.Popen(
    ["bash", "-lc", cmd],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

lines = []
line_times = []

pbar = tqdm(desc="FAO2HKB run", unit="line", leave=True)

try:
    for line in proc.stdout:
        dt = time.perf_counter() - t0
        lines.append(line)
        line_times.append(dt)

        pbar.update(1)
        tail = line.strip()
        if tail:
            pbar.set_postfix_str(tail[:80])
finally:
    pbar.close()

rc = proc.wait()
t1 = time.perf_counter()

out = "".join(lines)

# ---------------------------------------------------------------------
# 2) Fail fast if the CLI returned an error
# ---------------------------------------------------------------------
if rc != 0:
    print("---- OUTPUT TAIL ----")
    print(out[-4000:])
    raise RuntimeError(f"FAO2HKB run failed (return code {rc})")

# ---------------------------------------------------------------------
# 3) Clean stdout (remove ANSI escape sequences) for archival in the bundle
# ---------------------------------------------------------------------
ansi_re = re.compile(r"\x1b\[[0-9;]*[A-Za-z]")
clean = ansi_re.sub("", out).replace("\r", "")

# The final JSON emitted by the CLI must include these keys
REQUIRED = {"run_dir", "data_dir", "domain_jsonl", "series_jsonl", "records_jsonl"}

def extract_last_run_json(text: str):
    """
    Extract the LAST JSON object from stdout that contains REQUIRED keys.
    This is robust to:
      - multi-line JSON blocks,
      - earlier {..} dictionaries printed by libraries,
      - trailing text after JSON.
    """
    dec = json.JSONDecoder()
    idx = text.rfind("{")
    while idx != -1:
        try:
            obj, _end = dec.raw_decode(text[idx:])
            if isinstance(obj, dict) and REQUIRED.issubset(set(obj.keys())):
                return obj
        except Exception:
            pass
        idx = text.rfind("{", 0, idx)
    return None

run_out = extract_last_run_json(clean)
if run_out is None:
    print("---- OUTPUT TAIL ----")
    print(clean[-4000:])
    raise RuntimeError(f"Could not extract final run JSON containing keys={sorted(REQUIRED)}")

# ---------------------------------------------------------------------
# 4) Resolve CLI-emitted paths (handles relative paths from repo cwd)
# ---------------------------------------------------------------------
def resolve_path(p: str) -> str:
    """
    Resolve a path emitted by the CLI to an existing absolute path.
    Tries common bases:
      - absolute paths as-is,
      - current notebook cwd,
      - REPO_DIR (CLI cwd),
      - /content (Colab default).
    """
    if not p:
        return p
    p = str(p)

    if os.path.isabs(p) and os.path.exists(p):
        return p

    if os.path.exists(p):
        return os.path.abspath(p)

    cand = os.path.join(REPO_DIR, p)
    if os.path.exists(cand):
        return os.path.abspath(cand)

    cand = os.path.join("/content", p)
    if os.path.exists(cand):
        return os.path.abspath(cand)

    # Best-effort: return an absolute "guess" for debugging
    return os.path.abspath(os.path.join(REPO_DIR, p))

RUN_DIR    = resolve_path(run_out["run_dir"])
DATA_DIR   = resolve_path(run_out["data_dir"])
DOMAIN_JL  = resolve_path(run_out["domain_jsonl"])
SERIES_JL  = resolve_path(run_out["series_jsonl"])
RECORDS_JL = resolve_path(run_out["records_jsonl"])

print("\n✅ RUN_DIR    =", RUN_DIR)
print("✅ DOMAIN_JL  =", DOMAIN_JL)
print("✅ SERIES_JL  =", SERIES_JL)
print("✅ RECORDS_JL =", RECORDS_JL)

# ---------------------------------------------------------------------
# 5) Validate existence of the expected JSONL artifacts (reviewer safety)
# ---------------------------------------------------------------------
missing = [p for p in [DOMAIN_JL, SERIES_JL, RECORDS_JL] if not os.path.exists(p)]
if missing:
    print("\n[DEBUG] Some expected files are missing:")
    for p in missing:
        print("  -", p)

    print("\n[DEBUG] Notebook CWD:", os.getcwd())
    print("[DEBUG] REPO_DIR:", REPO_DIR)

    # Colab convenience diagnostics (safe on local if /content does not exist)
    if os.path.isdir("/content"):
        print("[DEBUG] /content contains:", sorted(os.listdir("/content"))[:80])

    repo_runs = os.path.join(REPO_DIR, "runs")
    if os.path.isdir(repo_runs):
        print("[DEBUG] REPO_DIR/runs contains:", sorted(os.listdir(repo_runs))[:30])

    if os.path.isdir(RUN_DIR):
        print("[DEBUG] RUN_DIR listing:", sorted(os.listdir(RUN_DIR))[:80])

    raise FileNotFoundError("Missing expected JSONL files. See debug output above.")

# ---------------------------------------------------------------------
# 6) Persist notebook-side evidence inside RUN_DIR/work/execution/
#    (These files are later included in the Zenodo bundle.)
# ---------------------------------------------------------------------
WORK_DIR = os.path.join(RUN_DIR, "work")
EXEC_DIR = os.path.join(WORK_DIR, "execution")
os.makedirs(EXEC_DIR, exist_ok=True)

stdout_path = os.path.join(EXEC_DIR, "cli_stdout.log")
with open(stdout_path, "w", encoding="utf-8") as f:
    f.write(clean)

run_out_path = os.path.join(EXEC_DIR, "cli_run_out.json")
with open(run_out_path, "w", encoding="utf-8") as f:
    json.dump(run_out, f, indent=2)

timings = {
    "total_wall_seconds": float(t1 - t0),
    "n_output_lines": int(len(lines)),
    "line_time_first_s": float(line_times[0]) if line_times else None,
    "line_time_last_s": float(line_times[-1]) if line_times else None,
}
timings_path = os.path.join(EXEC_DIR, "cli_timings.json")
with open(timings_path, "w", encoding="utf-8") as f:
    json.dump(timings, f, indent=2)

# ---------------------------------------------------------------------
# 7) Pipeline milestone timeline (if emitted by the CLI)
# ---------------------------------------------------------------------
exec_jsonl = run_out.get("execution_milestones_jsonl")
exec_json  = run_out.get("execution_milestones_json")
exec_sum   = run_out.get("execution_summary")

exec_jsonl_r = resolve_path(exec_jsonl) if exec_jsonl else None
exec_json_r  = resolve_path(exec_json)  if exec_json else None
exec_sum_r   = resolve_path(exec_sum)   if exec_sum else None

# ---------------------------------------------------------------------
# 8) Print a short, reviewer-friendly report
# ---------------------------------------------------------------------
print("\n========== TIMING REPORT ==========")
print(f"Total wall time : {timings['total_wall_seconds']:.2f}s")
print(f"Output lines    : {timings['n_output_lines']}")
print(f"Saved stdout    : {stdout_path}")
print(f"Saved run_out   : {run_out_path}")
print(f"Saved timings   : {timings_path}")

if exec_jsonl_r or exec_json_r or exec_sum_r:
    print("\n========== EXECUTION (PIPELINE) ==========")
    if exec_jsonl_r:
        print("✅ milestones.jsonl:", exec_jsonl_r, "| exists:", os.path.exists(exec_jsonl_r))
    if exec_json_r:
        print("✅ milestones.json :", exec_json_r,  "| exists:", os.path.exists(exec_json_r))
    if exec_sum_r:
        print("✅ summary.json    :", exec_sum_r,   "| exists:", os.path.exists(exec_sum_r))

    # Preview timeline (first 10 + last 5) to show that the pipeline progressed end-to-end
    try:
        events = []
        if exec_jsonl_r and os.path.exists(exec_jsonl_r):
            with open(exec_jsonl_r, "r", encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if line:
                        events.append(json.loads(line))
        elif exec_json_r and os.path.exists(exec_json_r):
            with open(exec_json_r, "r", encoding="utf-8") as f:
                events = json.load(f)

        if events:
            print("\nFirst 10 milestones:")
            for e in events[:10]:
                print(f"  [{e.get('t_seconds', 0):7.2f}s] {e.get('name')} meta={e.get('meta', {})}")

            if len(events) > 10:
                print("\nLast 5 milestones:")
                for e in events[-5:]:
                    print(f"  [{e.get('t_seconds', 0):7.2f}s] {e.get('name')} meta={e.get('meta', {})}")
        else:
            print("\n(⚠️ No events loaded from execution milestones.)")
    except Exception as e:
        print("\n(⚠️ Could not preview pipeline milestones:", type(e).__name__, str(e)[:180], ")")

    print("=========================================\n")
else:
    print("\n(ℹ️ No execution_* fields found in run_out. "
          "If you expected them, ensure the CLI emitted execution milestones.)")

print("==================================\n")


Running: cd "/content/fao2hkb" && python -m fao2hkb run --config "/content/config.resolved.yaml"


FAO2HKB run: 0line [00:00, ?line/s]


✅ RUN_DIR    = /content/fao2hkb/runs/FAO2HKB_20260107_181757
✅ DOMAIN_JL  = /content/fao2hkb/runs/FAO2HKB_20260107_181757/artifacts/data/DOMAIN.jsonl
✅ SERIES_JL  = /content/fao2hkb/runs/FAO2HKB_20260107_181757/artifacts/data/SERIES.jsonl
✅ RECORDS_JL = /content/fao2hkb/runs/FAO2HKB_20260107_181757/artifacts/data/RECORDS.jsonl

========== TIMING REPORT ==========
Total wall time : 3665.28s
Output lines    : 382
Saved stdout    : /content/fao2hkb/runs/FAO2HKB_20260107_181757/work/execution/cli_stdout.log
Saved run_out   : /content/fao2hkb/runs/FAO2HKB_20260107_181757/work/execution/cli_run_out.json
Saved timings   : /content/fao2hkb/runs/FAO2HKB_20260107_181757/work/execution/cli_timings.json

========== EXECUTION (PIPELINE) ==========
✅ milestones.jsonl: /content/fao2hkb/runs/FAO2HKB_20260107_181757/work/execution/milestones.jsonl | exists: True
✅ milestones.json : /content/fao2hkb/runs/FAO2HKB_20260107_181757/work/execution/milestones.json | exists: True
✅ summary.json    : /content/

## Diagnostics & visualizations (dataset characterization)

This step produces:
- **Series-level properties table** (`SERIES_properties.csv`)
- **Global summary table** (min/median/mean/std per property)
- **Distribution plots** (boxplots, global and per Domain_table)
- **Optional PCA maps** if embeddings are present in `SERIES.jsonl`

Why this matters (Data descriptor style):
- Properties summarize dataset shape and potential quality issues (coverage, zeros, nulls, etc.).
- PCA maps provide an interpretable view of semantic structure when embeddings exist.


In [ ]:
#@title 5) Generate diagnostics (properties + PCA if available)
"""
MDPI/Data descriptor note:
- This cell generates reviewer-friendly diagnostics from the produced JSONL artifacts.
- It does not modify the dataset; it only summarizes properties and (optionally) visualizes embeddings.
- Outputs are written under RUN_DIR/work/ so the Zenodo bundle remains self-contained.
"""

import os, re, json, inspect, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
from sklearn.decomposition import PCA

# ---------------------------------------------------------------------
# 0) Paths (all notebook outputs live under RUN_DIR/work/)
# ---------------------------------------------------------------------
WORK_DIR  = os.path.join(RUN_DIR, "work")
PLOTS_DIR = os.path.join(WORK_DIR, "plots", "diagnostics")
PROPS_DIR = os.path.join(WORK_DIR, "execution", "properties")

os.makedirs(PLOTS_DIR, exist_ok=True)
os.makedirs(PROPS_DIR, exist_ok=True)

# ---------------------------------------------------------------------
# 1) Config check (were embeddings intended?)
# ---------------------------------------------------------------------
with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    _cfg = yaml.safe_load(f)

EMB_ENABLED = bool((_cfg.get("embeddings") or {}).get("enabled", False))
print("Embeddings enabled in config:", EMB_ENABLED)

# ---------------------------------------------------------------------
# 2) Plot controls (reviewer-friendly defaults)
# ---------------------------------------------------------------------
SAMPLE_N = 10000  # @param {type:"integer"}  # max number of points used for PCA plots

# Legend controls
LEGEND_MAX_CATEGORIES = 10    # @param {type:"integer"}  # top-K categories kept; rest => "(other)"
LEGEND_HIDE = False          # @param {type:"boolean"}  # if True: do not draw legends (paper-friendly)

# Style controls (kept simple and stable)
PALETTE    = "tab20"  # @param {type:"string"}
POINT_SIZE = 10
ALPHA      = 1.0
LEGEND_COLS = 1       # @param {type:"integer"}  # legend columns
LEGEND_FONT_SIZE = 8  # @param {type:"integer"}  # compact legends

# ---------------------------------------------------------------------
# 3) Matplotlib compatibility helper (older versions use "labels")
# ---------------------------------------------------------------------
def ax_boxplot(ax, data, labels, **kwargs):
    sig = inspect.signature(ax.boxplot)
    if "tick_labels" in sig.parameters:
        return ax.boxplot(data, tick_labels=labels, **kwargs)
    else:
        return ax.boxplot(data, labels=labels, **kwargs)

def safe_name(s: str) -> str:
    """Filesystem-safe short names for plot filenames."""
    s = re.sub(r"[^a-zA-Z0-9_-]+", "_", str(s))[:120]
    return s.strip("_") or "x"

# ---------------------------------------------------------------------
# 4) Map domain_key -> L3 dims (Domain/Sub_domain/Domain_table)
# ---------------------------------------------------------------------
dk2dims = {}
for r in read_jsonl_iter(DOMAIN_JL):
    dk = (r.get("identity", {}) or {}).get("domain_key")
    dims = r.get("dims", {}) or {}
    if dk:
        dk2dims[dk] = dims

def _get_l3(dk: str):
    d = dk2dims.get(dk, {}) or {}
    return (
        d.get("Domain") or d.get("domain") or "",
        d.get("Sub_domain") or d.get("sub_domain") or "",
        d.get("Domain_table") or d.get("domain_table") or str(dk),
    )

# ---------------------------------------------------------------------
# 5) Build a properties DataFrame + collect embeddings (if present)
# ---------------------------------------------------------------------
rows = []
emb_list = []
dim_rows = []
missing_emb = 0
has_any_emb = False

for r in read_jsonl_iter(SERIES_JL):
    ident = r.get("identity", {}) or {}
    dims  = r.get("dims", {}) or {}
    props = r.get("properties", {}) or {}

    sk = ident.get("series_key")
    dk = ident.get("domain_key")
    if not sk or not dk:
        continue

    domain, sub_domain, domain_table = _get_l3(dk)

    row = {
        "series_key": sk,
        "domain_key": dk,
        "Domain": domain,
        "Sub_domain": sub_domain,
        "Domain_table": domain_table,
        "Area": dims.get("Area"),
        "Item": dims.get("Item"),
        "Element": dims.get("Element"),
        "Unit": dims.get("Unit"),
    }

    # Store properties as columns prefixed with prop__
    for k, v in props.items():
        row[f"prop__{k}"] = v

    # Optional embeddings in SERIES.jsonl (description.embedding)
    emb = (r.get("description", {}) or {}).get("embedding", None)
    if emb is None:
        missing_emb += 1
    else:
        has_any_emb = True
        try:
            v = np.asarray(emb, dtype=np.float32)
            if v.ndim == 1 and v.size > 0 and np.isfinite(v).all():
                emb_list.append(v)
                dim_rows.append({
                    "Domain": domain,
                    "Sub_domain": sub_domain,
                    "Domain_table": domain_table,
                    "Area": dims.get("Area"),
                    "Item": dims.get("Item"),
                    "Element": dims.get("Element"),
                    "Unit": dims.get("Unit"),
                })
        except Exception:
            # If a single embedding is malformed, we skip it (robust for reviewers).
            pass

    rows.append(row)

df_series_props = pd.DataFrame(rows)
print("✅ df_series_props:", df_series_props.shape)

SERIES_PROPS_CSV = os.path.join(PROPS_DIR, "SERIES_properties.csv")
df_series_props.to_csv(SERIES_PROPS_CSV, index=False)
print("✅ saved:", SERIES_PROPS_CSV)

# ---------------------------------------------------------------------
# 6) Detect numeric properties (enough coverage to be meaningful)
# ---------------------------------------------------------------------
prop_cols = [c for c in df_series_props.columns if c.startswith("prop__")]
df_num = df_series_props[prop_cols].apply(pd.to_numeric, errors="coerce")

numeric_cols = []
for c in prop_cols:
    nn = int(df_num[c].notna().sum())
    if nn >= 50 and (df_num[c].notna().mean() > 0.60):
        numeric_cols.append(c)

numeric_cols_sorted = sorted(numeric_cols, key=lambda c: df_num[c].notna().sum(), reverse=True)
print("Numeric properties:", [c.replace("prop__","") for c in numeric_cols_sorted])

# ---------------------------------------------------------------------
# 7) Global stats table (paper-friendly summary)
# ---------------------------------------------------------------------
stats_rows = []
for c in numeric_cols_sorted:
    v = pd.to_numeric(df_series_props[c], errors="coerce").dropna()
    if len(v) == 0:
        continue
    stats_rows.append({
        "property": c.replace("prop__",""),
        "n": int(v.shape[0]),
        "min": float(v.min()),
        "p25": float(v.quantile(0.25)),
        "median": float(v.median()),
        "mean": float(v.mean()),
        "p75": float(v.quantile(0.75)),
        "max": float(v.max()),
        "std": float(v.std(ddof=1)) if v.shape[0] > 1 else 0.0,
    })

df_stats = pd.DataFrame(stats_rows).sort_values("n", ascending=False)
STATS_CSV = os.path.join(PROPS_DIR, "L2_properties_global_stats.csv")
df_stats.to_csv(STATS_CSV, index=False)
print("✅ saved:", STATS_CSV)

# ---------------------------------------------------------------------
# 8) Boxplots (global grid)
# ---------------------------------------------------------------------
if numeric_cols_sorted:
    P = len(numeric_cols_sorted)
    ncols = 4
    nrows = math.ceil(P / ncols)

    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3 * nrows), squeeze=False)

    used = 0
    for i, c in enumerate(numeric_cols_sorted):
        v = pd.to_numeric(df_series_props[c], errors="coerce").dropna()
        if len(v) == 0:
            continue

        ax = axes.flat[i]
        prop_name = c.replace("prop__", "")
        ax_boxplot(ax, [v.values], labels=[prop_name], showfliers=False)
        ax.tick_params(axis="x", labelrotation=45)
        ax.set_ylabel("value")
        ax.set_title(prop_name)
        used += 1

    for j in range(used, nrows * ncols):
        axes.flat[j].set_visible(False)

    fig.suptitle("Series properties — global distributions (1 subplot per property)", fontsize=12)
    fig.tight_layout(rect=[0, 0, 1, 0.96])

    out_global_grid = os.path.join(PLOTS_DIR, "props__box__global__grid.png")
    fig.savefig(out_global_grid, dpi=200)
    plt.close(fig)
    print("✅ global grid boxplots:", out_global_grid)
else:
    print("ℹ️ No numeric properties found (boxplots skipped).")

# ---------------------------------------------------------------------
# 9) Boxplots per Domain_table (helps compare tables with different scales)
# ---------------------------------------------------------------------
tables = df_series_props["Domain_table"].dropna().astype(str).unique().tolist()
for table in sorted(tables):
    df_t = df_series_props[df_series_props["Domain_table"].astype(str) == table]

    cols_for_table = []
    for c in numeric_cols_sorted:
        v = pd.to_numeric(df_t[c], errors="coerce").dropna()
        if len(v) > 0:
            cols_for_table.append(c)

    if not cols_for_table:
        continue

    P_t = len(cols_for_table)
    ncols = 4
    nrows = math.ceil(P_t / ncols)

    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3 * nrows), squeeze=False)

    used = 0
    for i, c in enumerate(cols_for_table):
        v = pd.to_numeric(df_t[c], errors="coerce").dropna()
        if len(v) == 0:
            continue

        ax = axes.flat[i]
        prop_name = c.replace("prop__", "")
        ax_boxplot(ax, [v.values], labels=[prop_name], showfliers=False)
        ax.tick_params(axis="x", labelrotation=45)
        ax.set_ylabel("value")
        ax.set_title(prop_name)
        used += 1

    for j in range(used, nrows * ncols):
        axes.flat[j].set_visible(False)

    fig.suptitle(f"Series properties by Domain_table — {table}", fontsize=12)
    fig.tight_layout(rect=[0, 0, 1, 0.96])

    out_table_grid = os.path.join(PLOTS_DIR, f"props__box__by_table__grid__{safe_name(table)}.png")
    fig.savefig(out_table_grid, dpi=200)
    plt.close(fig)
    print("✅ per-table grid boxplots:", out_table_grid)

# ---------------------------------------------------------------------
# 10) PCA maps (only if embeddings exist)
# ---------------------------------------------------------------------
print("Embeddings present in SERIES.jsonl:", has_any_emb, "| missing:", missing_emb)

if EMB_ENABLED and has_any_emb and len(emb_list) >= 10:
    X = np.vstack(emb_list).astype(np.float32)
    df_dims = pd.DataFrame(dim_rows)

    # Subsample to keep figures readable and fast for reviewers
    rng = np.random.default_rng(int(SEED))
    n = X.shape[0]
    if n > int(SAMPLE_N) > 0:
        idx = rng.choice(n, size=int(SAMPLE_N), replace=False)
        Xs = X[idx]
        ds = df_dims.iloc[idx].reset_index(drop=True)
    else:
        Xs = X
        ds = df_dims.reset_index(drop=True)

    # PCA is deterministic given SEED; we store explained variance for reporting
    pca = PCA(n_components=2, random_state=int(SEED))
    Z = pca.fit_transform(Xs)

    evr = pca.explained_variance_ratio_.tolist()
    meta_path = os.path.join(PLOTS_DIR, "emb__pca__meta.json")
    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump({
            "method": "pca",
            "n_total_embeddings": int(n),
            "n_used": int(Xs.shape[0]),
            "explained_variance_ratio": [float(evr[0]), float(evr[1])],
            "seed": int(SEED),
            "sample_n": int(SAMPLE_N),
        }, f, indent=2)
    print("✅", meta_path)

    DIMS_L3 = ["Domain", "Sub_domain", "Domain_table"]
    DIMS_L2 = ["Area", "Item", "Element", "Unit"]
    ALL_DIMS = DIMS_L3 + DIMS_L2

    def _topk_other_labels(s: pd.Series, topk: int):
        """
        Keep top-k labels (by frequency) and collapse the rest into "(other)".
        Returns:
          - lab: mapped labels (top-k + other)
          - cats_order: category order for stable legends
          - counts: counts after collapsing (for reporting)
        """
        s = s.fillna("").astype(str)
        vc = s.value_counts(dropna=False)

        tmp = vc.reset_index()
        tmp.columns = ["cat", "count"]
        tmp = tmp.sort_values(["count", "cat"], ascending=[False, True])

        top = tmp.head(int(topk))["cat"].tolist()
        keep = set(top)
        lab = s.where(s.isin(keep), other="(other)")

        vc2 = lab.value_counts(dropna=False)
        tmp2 = vc2.reset_index()
        tmp2.columns = ["cat", "count"]
        tmp2 = tmp2.sort_values(["count", "cat"], ascending=[False, True])

        cats_order = tmp2["cat"].tolist()
        return lab, cats_order, vc2.to_dict()

    def plot_pca_by_dim(dim: str):
        """
        Generate a single PCA scatter plot colored by one categorical dimension.
        Legend size is controlled by LEGEND_MAX_CATEGORIES (top-k).
        """
        lab, cats_order, counts = _topk_other_labels(ds[dim], topk=int(LEGEND_MAX_CATEGORIES))

        # Save counts for transparency (reviewers can audit what "other" contains)
        counts_path = os.path.join(PLOTS_DIR, f"emb__pca__counts__{safe_name(dim)}.json")
        with open(counts_path, "w", encoding="utf-8") as f:
            json.dump({
                "dim": dim,
                "legend_topk": int(LEGEND_MAX_CATEGORIES),
                "counts_after_collapse": {k: int(v) for k, v in counts.items()},
            }, f, indent=2)

        n_cats = len(cats_order)
        cmap = plt.get_cmap(PALETTE, n_cats)

        fig, ax = plt.subplots(figsize=(9, 7))

        # Plot larger groups first so smaller groups remain visible
        for i, cat in enumerate(cats_order):
            m = (lab.values == cat)
            if not np.any(m):
                continue
            ax.scatter(
                Z[m, 0], Z[m, 1],
                s=int(POINT_SIZE),
                alpha=float(ALPHA),
                color=cmap(i),
                edgecolors="none",
                linewidths=0.0,
                rasterized=True,
                label=f"{cat} ({int(counts.get(cat, 0))})"
            )

        ax.set_title(
            f"Embeddings PCA (2D) — colored by {dim} "
            f"(legend: top {int(LEGEND_MAX_CATEGORIES)} + other)"
        )
        ax.set_xlabel("PC1")
        ax.set_ylabel("PC2")
        ax.grid(True, alpha=0.15)

        # Legend policy: single boolean switch (reviewer-friendly)
        if not bool(LEGEND_HIDE):
            ax.legend(
                loc="best",
                fontsize=int(LEGEND_FONT_SIZE),
                frameon=True,
                ncol=max(1, int(LEGEND_COLS)),
            )

        fig.tight_layout()

        out = os.path.join(PLOTS_DIR, f"emb__pca__color__{safe_name(dim)}.png")
        fig.savefig(out, dpi=200)
        plt.close(fig)
        return out

    saved = []
    for dim in ALL_DIMS:
        try:
            out_png = plot_pca_by_dim(dim)
            saved.append(out_png)
            print("✅", out_png)
        except Exception as e:
            print(f"⚠️ PCA plot failed for dim={dim}: {type(e).__name__}: {str(e)[:160]}")

    print("\n✅ Total PCA plots saved:", len(saved))

else:
    if not EMB_ENABLED:
        print("ℹ️ Embeddings disabled → PCA skipped (properties were generated).")
    else:
        print("⚠️ Embeddings enabled but not enough embeddings found → PCA skipped.")

print("✅ plots folder     :", PLOTS_DIR)
print("✅ properties folder:", PROPS_DIR)


Embeddings enabled in config: True
✅ df_series_props: (446482, 20)
✅ saved: /content/fao2hkb/runs/FAO2HKB_20260107_181757/work/execution/properties/SERIES_properties.csv
Numeric properties: ['year_min', 'year_max', 'n_members', 'zeros', 'nulls', 'min', 'max', 'mean', 'median', 'variance', 'entropy']
✅ saved: /content/fao2hkb/runs/FAO2HKB_20260107_181757/work/execution/properties/L2_properties_global_stats.csv
✅ global grid boxplots: /content/fao2hkb/runs/FAO2HKB_20260107_181757/work/plots/diagnostics/props__box__global__grid.png
✅ per-table grid boxplots: /content/fao2hkb/runs/FAO2HKB_20260107_181757/work/plots/diagnostics/props__box__by_table__grid__Crops_and_livestock_products.png
✅ per-table grid boxplots: /content/fao2hkb/runs/FAO2HKB_20260107_181757/work/plots/diagnostics/props__box__by_table__grid__Emissions_from_Crops.png
✅ per-table grid boxplots: /content/fao2hkb/runs/FAO2HKB_20260107_181757/work/plots/diagnostics/props__box__by_table__grid__Emissions_from_Drained_organic_soil

## Human-readable demo (10 random time-series)

This section is meant for readers and reviewers:
- It selects **10 random series** from `SERIES.jsonl`.
- It prints “cards” connecting:
  - **L3** context (Domain/Sub-domain/Domain-table),
  - **L2** series identity (Area/Item/Element/Unit),
  - **L1** records (Year–Value points).
- It also saves one plot per series (time-series + value distribution).

Outputs are stored under `RUN_DIR/work/`.


In [ ]:
#@title 6) Produce 10-series demo (cards + plots)
"""
MDPI/Data descriptor note:
- This cell is a human-readable demonstration only.
- It does not modify the dataset; it samples and visualizes a small subset.
- Outputs are written under RUN_DIR/work/ so the Zenodo bundle remains self-contained.
"""

import os, json
import numpy as np
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------
# 0) User controls (reviewer-friendly defaults)
# ---------------------------------------------------------------------
RANDOM_SERIES_N = 10   # @param {type:"integer"}  # number of series to sample and display

# Keep all points for transparency (reviewers can see the full series)
HIST_BINS = None       # None => matplotlib "auto" (stable, no tuning required)

# ---------------------------------------------------------------------
# 1) Output paths (all notebook outputs live under RUN_DIR/work/)
# ---------------------------------------------------------------------
WORK_DIR = os.path.join(RUN_DIR, "work")
os.makedirs(WORK_DIR, exist_ok=True)

PLOTS_DIR = os.path.join(WORK_DIR, "plots", "random_series")
os.makedirs(PLOTS_DIR, exist_ok=True)

CARDS_TXT_PATH  = os.path.join(WORK_DIR, "random_series_cards.txt")
CARDS_JSON_PATH = os.path.join(WORK_DIR, "random_series_cards.json")

# ---------------------------------------------------------------------
# 2) Sample series uniformly from SERIES.jsonl
# ---------------------------------------------------------------------
# We use reservoir sampling so this works even when SERIES.jsonl is large.
sample_series = reservoir_sample_series(SERIES_JL, k=int(RANDOM_SERIES_N), seed=int(SEED))

# Build a lookup by series_key (and remove accidental duplicates)
keys = []
ser_by_key = {}
for r in sample_series:
    sk = (r.get("identity", {}) or {}).get("series_key")
    if sk:
        keys.append(sk)
        ser_by_key[sk] = r

keys = list(dict.fromkeys(keys))
if not keys:
    raise RuntimeError("No series keys sampled from SERIES.jsonl (unexpected).")

# ---------------------------------------------------------------------
# 3) Load L3 metadata (DOMAIN.jsonl) for context
# ---------------------------------------------------------------------
# Reviewers can see which Domain/Sub-domain/Domain-table each series belongs to.
domains = load_domains(DOMAIN_JL)
dom_by_key = {}
for d in domains:
    dk = (d.get("identity", {}) or {}).get("domain_key")
    if dk:
        dom_by_key[dk] = d.get("dims", {}) or {}

# ---------------------------------------------------------------------
# 4) Fetch L1 records for the sampled series (RECORDS.jsonl)
# ---------------------------------------------------------------------
# We scan the JSONL once and stop when all sampled series are found.
rec_by_key = scan_records_for_keys(RECORDS_JL, set(keys))

# ---------------------------------------------------------------------
# 5) Helpers (robust parsing + plotting)
# ---------------------------------------------------------------------
def _safe_float(x):
    """Parse numeric values safely; return None if invalid."""
    try:
        v = float(x)
        if np.isfinite(v):
            return v
    except Exception:
        pass
    return None

def _extract_xy(records_row):
    """
    Extract (Year, Value) arrays from a RECORDS.jsonl row.
    We sort by Year for plotting and printing.
    """
    data = (records_row or {}).get("data", []) or []
    ys, vs = [], []

    for d in data:
        yf = _safe_float(d.get("Year", None))
        vf = _safe_float(d.get("Value", None))
        if yf is None or vf is None:
            continue
        ys.append(yf)
        vs.append(vf)

    # If we have fewer than 2 points, plotting is not meaningful
    if len(ys) < 2:
        return np.array([]), np.array([])

    y = np.asarray(ys, dtype=float)
    v = np.asarray(vs, dtype=float)
    order = np.argsort(y)
    return y[order], v[order]

def _build_header(dims_l3, dims_l2):
    """Compact header used for plot titles and card metadata."""
    return (
        "FAOSTAT series | "
        f"Domain: {dims_l3.get('Domain','')} | "
        f"Sub-domain: {dims_l3.get('Sub_domain','')} | "
        f"Domain-table: {dims_l3.get('Domain_table','')} | "
        f"Area: {dims_l2.get('Area','')} | "
        f"Item: {dims_l2.get('Item','')} | "
        f"Element: {dims_l2.get('Element','')} | "
        f"Unit: {dims_l2.get('Unit','')}"
    )

def plot_two_panel(y, v, header, unit, out_png):
    """
    Make a 2-panel plot:
      (left)  time-series over years
      (right) value distribution (histogram)
    """
    if len(y) < 2 or len(v) < 2:
        return False

    # Matplotlib "auto" bins is reviewer-friendly: it avoids tuning.
    bins = "auto" if HIST_BINS is None else int(HIST_BINS)

    fig, axes = plt.subplots(
        1, 2, figsize=(14, 4),
        gridspec_kw={"width_ratios": [3.2, 1.4]}
    )
    ax_ts, ax_hist = axes

    ax_ts.plot(y, v, marker="o")
    ax_ts.set_title("Time-series")
    ax_ts.set_xlabel("Year")
    ax_ts.set_ylabel(unit or "Value")
    ax_ts.grid(True, alpha=0.25)

    ax_hist.hist(v, bins=bins, orientation="horizontal")
    ax_hist.set_title("Value distribution")
    ax_hist.set_xlabel("Frequency")
    ax_hist.set_ylabel("Value")
    ax_hist.grid(True, axis="x", alpha=0.25)

    fig.suptitle(header, fontsize=10)
    fig.tight_layout(rect=[0, 0, 1, 0.90])
    fig.savefig(out_png, dpi=200)
    plt.close(fig)
    return True

# ---------------------------------------------------------------------
# 6) Build ASCII cards + plots + JSON manifest
# ---------------------------------------------------------------------
lines = []
manifest = []

print("\n========== RANDOM SERIES: L3 → L2 → L1 cards ==========\n")
lines.append("========== RANDOM SERIES: L3 → L2 → L1 cards ==========\n\n")

for idx, sk in enumerate(keys, start=1):
    srow = ser_by_key.get(sk, {})
    ident = srow.get("identity", {}) or {}
    dk = ident.get("domain_key")

    # L3 context (domain-level descriptors)
    dims_l3 = dom_by_key.get(dk, {}) or {}

    # L2 identity (series dims like Area/Item/Element/Unit)
    dims_l2 = srow.get("dims", {}) or {}

    # Optional series properties (useful diagnostics)
    props = srow.get("properties", {}) or {}

    # L1 records stream (Year/Value points)
    rrow = rec_by_key.get(sk, {}) or {}
    y, v = _extract_xy(rrow)

    n_pts = int(len(y))
    y_min = int(np.min(y)) if n_pts else None
    y_max = int(np.max(y)) if n_pts else None

    # Keep all points in the report for transparency
    all_points = [{"Year": int(round(a)), "Value": float(b)} for a, b in zip(y, v)]

    def _emit(s=""):
        print(s)
        lines.append(s + "\n")

    _emit("=" * 80)
    _emit(f"[{idx}/{len(keys)}] series_key={sk}")
    _emit(f"  domain_key={dk}")

    _emit("\n  L3 — Domain context")
    _emit(f"    Domain      : {dims_l3.get('Domain', '')}")
    _emit(f"    Sub_domain  : {dims_l3.get('Sub_domain', '')}")
    _emit(f"    Domain_table: {dims_l3.get('Domain_table', '')}")

    _emit("\n  L2 — Series identity")
    _emit(f"    Area        : {dims_l2.get('Area', '')}")
    _emit(f"    Item        : {dims_l2.get('Item', '')}")
    _emit(f"    Element     : {dims_l2.get('Element', '')}")
    _emit(f"    Unit        : {dims_l2.get('Unit', '')}")

    # Optional: show additional dims (if present)
    core_keys = {"Area", "Item", "Element", "Unit"}
    extra_dims = {
        k: v for k, v in (dims_l2.items() if isinstance(dims_l2, dict) else [])
        if k not in core_keys and v not in (None, "")
    }
    if extra_dims:
        _emit("    --- Extra dims ---")
        for k in sorted(extra_dims.keys()):
            _emit(f"    {k:12s}: {extra_dims[k]}")

    # A small selection of properties that is typically meaningful for reviewers
    keep_prop_keys = ["n_members", "year_min", "year_max", "zeros", "nulls", "trainable"]
    for k in keep_prop_keys:
        if k in props:
            _emit(f"    {k:10s}: {props.get(k)}")

    _emit("\n  L1 — Records (Year → Value)")
    if n_pts:
        _emit(f"    Years range : {y_min}–{y_max} ({n_pts} points)")
        _emit("    Points (ALL):")
        for p in all_points:
            _emit(f"      {p['Year']}: {p['Value']}")
    else:
        _emit("    (no valid Year/Value pairs)")

    # Plot (2-panel)
    header = _build_header(dims_l3, dims_l2)
    unit = dims_l2.get("Unit", "Value")

    out_png = os.path.join(PLOTS_DIR, f"series_{idx:02d}__2panel.png")
    saved_plot = plot_two_panel(y, v, header, unit, out_png)
    _emit(f"\n    Plot saved  : {out_png}" if saved_plot else "\n    Plot skipped: no data")

    # Manifest entry (machine-readable summary)
    manifest.append({
        "series_key": sk,
        "domain_key": dk,
        "header": header,
        "dims_l3": dims_l3,
        "dims_l2": dims_l2,
        "properties": {k: props.get(k) for k in keep_prop_keys},
        "n_points": n_pts,
        "year_min": y_min,
        "year_max": y_max,
        "points": all_points,
        "plot_path": out_png if saved_plot else None,
    })

    _emit("")

# ---------------------------------------------------------------------
# 7) Persist outputs (plain-text report + JSON manifest)
# ---------------------------------------------------------------------
with open(CARDS_TXT_PATH, "w", encoding="utf-8") as f:
    f.writelines(lines)

with open(CARDS_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

print("\n============================================================\n")
print("✅ cards (txt):", CARDS_TXT_PATH)
print("✅ cards (json):", CARDS_JSON_PATH)
print("✅ plots      :", PLOTS_DIR)



========== RANDOM SERIES: L3 → L2 → L1 cards ==========

[1/10] series_key=sha256:50a6ddc8d396517c463d510d38eca3955d8fc035e9e5a80b6e2fe6aa6a2d4977
  domain_key=sha256:885de5171eed0eb90b84b60b1495e810c247bff85fa9dd3c9d93f2572159d638

  L3 — Domain context
    Domain      : Climate Change: Agrifood systems emissions
    Sub_domain  : Farm gate
    Domain_table: Emissions from Livestock

  L2 — Series identity
    Area        : Low Income Food Deficit Countries (LIFDCs)
    Item        : Swine, market
    Element     : Manure applied to soils (Indirect emissions N2O)
    Unit        : kt
    --- Extra dims ---
    Area_code   : 5815
    Element_code: 72361
    Item_code   : 1049
    n_members : 65
    year_min  : 1961
    year_max  : 2050
    zeros     : 0
    nulls     : 0

  L1 — Records (Year → Value)
    Years range : 1961–2050 (65 points)
    Points (ALL):
      1961: 0.4124
      1962: 0.4335
      1963: 0.4538
      1964: 0.4656
      1965: 0.4734
      1966: 0.494
      1967: 0.5

## Bundle for Zenodo (single ZIP)

This final step creates **one ZIP file** intended for Zenodo upload.

Included:
- `config.resolved.yaml` (exact config used)
- `repo_commit.txt` (code provenance: repo + commit)
- run artifacts (`artifacts/`)
- reproducibility evidence (`work/execution/`)
- plots and demo cards (`work/plots/`, `work/random_series_cards.*`)
- optional cached raw inputs (`raw/`, if present)

This ZIP is the “data product package” for reviewers.


In [ ]:
#@title 8) Bundle outputs for Zenodo (single ZIP)
"""
MDPI/Data descriptor note:
- This ZIP is the single artifact intended for Zenodo upload.
- It contains both the data product (artifacts) and the evidence (config/commit/logs/plots).
- The bundle is self-contained: a reviewer can inspect inputs, outputs, and provenance offline.
"""

import zipfile
from pathlib import Path

# ---------------------------------------------------------------------
# 0) Resolve run folder + choose a stable bundle name
# ---------------------------------------------------------------------
run_dir = Path(RUN_DIR)
assert run_dir.exists(), f"RUN_DIR not found: {RUN_DIR}"

bundle_name = f"fao2hkb_bundle__{run_dir.name}.zip"
zip_path = str(Path.cwd() / bundle_name)

# ---------------------------------------------------------------------
# 1) What we include (explicit list = reviewer-friendly transparency)
# ---------------------------------------------------------------------
# Notes:
# - Some files may be optional depending on your pipeline configuration (we skip missing paths quietly).
# - Keeping "raw/" is useful when reviewers want to confirm exact inputs, but it can increase ZIP size.
paths_to_include = [
    Path(CONFIG_PATH),                 # notebook-side resolved config (exact parameters used)
    Path.cwd() / "repo_commit.txt",    # code provenance (repo URL + ref + commit hash)

    run_dir / "manifest.json",         # pipeline manifest (if present)
    run_dir / "config.resolved.yaml",  # pipeline-copied config (if present)

    run_dir / "artifacts",             # data product (DOMAIN/SERIES/RECORDS + related outputs)
    run_dir / "work",                  # evidence: logs, timings, plots, demo cards
    run_dir / "raw",                   # optional cached inputs (if pipeline stores them here)
]

# Include tarballs if they exist (safe even if tar_gz was disabled)
for tgz in run_dir.glob("*.tar.gz"):
    paths_to_include.append(tgz)

# ---------------------------------------------------------------------
# 2) ZIP layout policy
# ---------------------------------------------------------------------
# We store run outputs under their natural folder name (typically: runs/<RUN_ID>/...).
# CONFIG_PATH and repo_commit.txt may live outside the runs/ tree; those are added at ZIP root.
ROOT_PREFIX = run_dir.parent  # typically ".../runs"

def add_path_to_zip(zf: zipfile.ZipFile, path: Path, root_prefix: Path) -> None:
    """
    Add files/dirs to a ZIP using stable, readable relative paths.

    Rules:
    - If the path lives under root_prefix, store it as a relative path (e.g., <RUN_ID>/work/...).
    - If it lives outside root_prefix (e.g., config.resolved.yaml in notebook cwd),
      store it at ZIP root by filename.
    - Missing optional paths are ignored quietly to avoid noise for reviewers.
    """
    if path.is_dir():
        for f in path.rglob("*"):
            if not f.is_file():
                continue
            try:
                arcname = f.relative_to(root_prefix)
            except ValueError:
                arcname = Path(f.name)
            zf.write(f, arcname.as_posix())

    elif path.is_file():
        try:
            arcname = path.relative_to(root_prefix)
        except ValueError:
            arcname = Path(path.name)
        zf.write(path, arcname.as_posix())

    else:
        # Quietly ignore missing optional paths
        return

# ---------------------------------------------------------------------
# 3) Create ZIP
# ---------------------------------------------------------------------
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in paths_to_include:
        add_path_to_zip(zf, p, ROOT_PREFIX)

print("✅ ZIP created:", zip_path)

# ---------------------------------------------------------------------
# 4) Optional: auto-download in Google Colab
# ---------------------------------------------------------------------
try:
    from google.colab import files
    files.download(zip_path)
except Exception:
    # In local runs (or restricted envs), reviewers can just download manually.
    print("ℹ️ Auto-download unavailable. Download manually from:", zip_path)

✅ ZIP created: /content/fao2hkb_bundle__FAO2HKB_20260107_181757.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>